# Agent Identity and Authorization

## Scenario: authorize an incident adviser without sharing user/admin credentials

The Acme incident adviser gets a distinct workload identity and a short-lived capability to read checkout status. It cannot cross tenants, broaden scope, use a standing secret, or execute high-impact work without approval.

**Safety boundary:** authorization is evaluated at the resource boundary; a model cannot grant itself capability.

![Delegated agent identity and authorization](../../../assets/agent-identity-authorization.svg)

Separate user identity, application issuer, agent non-human identity, and delegated authority. Bind a capability to tenant, resource, action, purpose, audience, expiry, and approval. Preserve an auditable chain of delegation.

In [1]:
from pathlib import Path
import sys
TOPIC=Path.cwd()
if not (TOPIC/'lab.py').exists(): TOPIC=Path.cwd()/'curriculum'/'enterprise-agent'/'11-agent-identity-authorization'
sys.path.insert(0,str(TOPIC))
from lab import Capability, authorize
cap=Capability('agent:incident-adviser','acme','read-status','checkout',expires=10)
assert authorize(cap,1,'acme','read-status')=='allow'
assert authorize(cap,1,'globex','read-status')=='deny'
assert authorize(cap,11,'acme','read-status')=='deny'
print(cap.audit)

['allow:delegated-capability', 'deny:scope', 'deny:expired']


## Production controls

Use OAuth/OIDC federation and workload/non-human identity where appropriate; validate issuer, audience, signature, expiry and claims using current provider documentation. Exchange identity for short-lived resource-scoped credentials. Enforce least privilege, tenant/purpose binding, approval, idempotency, rate/budget, auditing, rotation, revocation, and peer authentication.

**Exercises:** add an approval-gated restart capability; test a confused-deputy request; model agent-to-agent handoff scopes; and design revocation/rotation telemetry.

References: [OAuth security BCP](https://datatracker.ietf.org/doc/html/draft-ietf-oauth-security-topics), [OIDC Core](https://openid.net/specs/openid-connect-core-1_0.html), [SPIFFE](https://spiffe.io/docs/latest/spiffe-about/overview/).